[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_65_Phase7_AgentBench_Kickoff.ipynb)

# Lesson 65 — Phase 7 Kickoff: `agent-bench` as a Second Flagship OSS Tool

Welcome to **Phase 7**. You just finished Phase 6 by shipping `paper-distiller` — a real, installable, CI-gated, launch-ready OSS package (9 lessons, L56–L64).

Today we start Phase 7 by asking a harder question than "can I build one good project?": **can I build a second one, from a component I already wrote, and prove it generalizes?**

In Lesson 61 you built `agent-bench` — a benchmark harness (Task → Environment → Agent → Trajectory → Scorer → Aggregate) that could score *any* agent, not just paper-distiller. You even proved this by benchmarking a stubbed `PaperDistillerAgent` alongside a chat agent and a scripted mock, all through the same interface. That was the tell: `agent-bench` was never really "part of" paper-distiller's lesson — it was already its own tool, just living inside a notebook.

This lesson repackages it as a standalone, pip-installable, extensible library with a plugin registry and a CLI — the second flagship repo for your portfolio.

## Phase 7 Roadmap (tentative — adapts as we go)

| # | Lesson | Focus |
|---|--------|-------|
| 65 | **Phase 7 Kickoff** | Repackage `agent-bench` as a standalone OSS library + plugin registry |
| 66 | Custom Environments | Write a `ShellEnv` (Terminal-Bench-shaped) + third-party environment plugin pattern |
| 67 | Suite Format & Leaderboards | YAML/JSON task-suite spec, multi-agent leaderboard table, HTML report |
| 68 | CLI + PyPI Packaging | `agent-bench run suite.yaml --agent my_agent` installable CLI (mirrors L58) |
| 69 | CI Gates & Contamination | Nightly benchmark CI, regression gate, leak detection at scale (extends L61 §15) |
| 70 | OSS Growth & Launch | README/badges/CONTRIBUTING/launch (mirrors L60/L64, faster this time) |

This roadmap is a starting point, not a contract — homework answers and questions you ask along the way can expand or reorder it, same as Phase 6 evolved from your Q&A. The one rule that doesn't change: **each lesson still has to be incrementally buildable on the last.**

**Why `agent-bench` over the other two L64 options (7A depth-over-breadth, 7C research literacy):** you already validated the hardest part — the harness's Task/Environment/Agent/Scorer contract genuinely generalizes across 3+ agent types (L61 §13). Turning proven-general code into a second shippable repo is a smaller, faster win than starting research-literacy content from zero, and it directly compounds your portfolio story: *"I didn't just build one agent, I built the tool other people use to evaluate agents."* That's a distinct, valuable niche.

## Concept: "a demo inside a notebook" vs. "a library other code depends on"

`agent-bench` already *worked* in L61. So what's actually changing today?

| Dimension | L61 (notebook demo) | Today (standalone library) |
|---|---|---|
| **Extensibility** | New environment/agent/scorer = edit the notebook's own cells | New environment/agent/scorer = **register a plugin**, zero edits to core |
| **Distribution** | Lives in one `.ipynb`, not importable elsewhere | `pip install agent-bench`, importable from any project (including `paper-distiller` itself) |
| **Discovery** | You have to know it exists and open the notebook | `agent-bench list-environments` / entry-point discovery |
| **Coupling** | Environments/agents/scorers are just Python names in one namespace | Registries decouple "the harness" from "the things it benchmarks" |
| **Failure mode if done wrong** | N/A — it's a demo | Core harness silently breaks when a third party's plugin misbehaves (needs default-deny / isolation, same lesson as L63's tool permission gate) |

The core technical addition today is the **plugin registry pattern** — the same idea behind pytest fixtures, Flask extensions, and `sklearn`'s estimator API: a small, stable core interface, plus a registration mechanism that lets anyone extend the system without forking it.

In [ ]:
# Setup — lightweight deps only. Reuses L61's harness shape; no chromadb/sentence-transformers needed today.
!pip install -q anthropic pydantic rich typer nest_asyncio 2>/dev/null

import os, sys, json, time, random, asyncio, math
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, Optional, Protocol
from pydantic import BaseModel, Field
from rich.console import Console
from rich.table import Table

console = Console(force_jupyter=False, no_color=True, highlight=False)  # L64 pitfall fix applied by default

import nest_asyncio
nest_asyncio.apply()

# Same graceful-degrade pattern as every prior lesson: works with zero setup, upgrades if a key is present.
HAVE_API_KEY = False
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
    if api_key:
        os.environ["ANTHROPIC_API_KEY"] = api_key
        HAVE_API_KEY = True
except Exception:
    HAVE_API_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))

print(f"HAVE_API_KEY = {HAVE_API_KEY}  (today's lesson runs fully offline either way — the harness itself needs no LLM calls)")


## Package architecture

```
agent_bench/
├── __init__.py          # public API: Task, Environment, Agent, Scorer, run_benchmark, register_*
├── core.py              # Task, TrajectoryStep, Trajectory, TaskResult, BenchmarkRunner, pass_at_k
├── registry.py          # ENVIRONMENT_REGISTRY / AGENT_REGISTRY / SCORER_REGISTRY + decorators
├── environments/
│   ├── calc.py          # CalcEnv        (built-in, self-registers on import)
│   ├── files.py         # FileEnv        (built-in, self-registers on import)
│   └── knowledge.py     # KnowledgeEnv   (built-in, self-registers on import)
├── agents/
│   ├── mock.py          # MockAgent
│   └── claude_tool.py   # ClaudeToolAgent
├── scorers/
│   └── builtin.py       # exact_match / unit_test / llm_judge
├── suites/
│   └── loader.py        # load_suite() — YAML/JSON task-suite spec → list[Task]
└── cli.py               # Typer CLI: run / list-environments / list-agents / list-scorers
```

The key design decision: **`core.py` never imports anything from `environments/`, `agents/`, or `scorers/`.** Those modules import *from* core and register themselves into it. Core stays stable; plugins come and go.

In [ ]:
# core.py — recreated from L61, unchanged in shape (this contract already proved itself, don't fix what isn't broken)

class Task(BaseModel):
    id: str
    category: str
    difficulty: str  # easy | medium | hard
    prompt: str
    env_name: str
    scorer_name: str
    metadata: dict = Field(default_factory=dict)

@dataclass
class TrajectoryStep:
    role: str          # "agent" | "environment"
    content: str
    tool_calls: list = field(default_factory=list)

@dataclass
class Trajectory:
    task_id: str
    steps: list = field(default_factory=list)
    final_state: Any = None
    cost_usd: float = 0.0
    elapsed_s: float = 0.0

@dataclass
class TaskResult:
    task_id: str
    passed: bool
    score: float
    trajectory: Trajectory
    error: Optional[str] = None

def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased pass@k estimator (Chen et al. 2021). n=attempts, c=correct, k=budget."""
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

print("core.py contract: Task / TrajectoryStep / Trajectory / TaskResult / pass_at_k — unchanged from L61")


## The new piece: a plugin registry

In L61, `CalcEnv`, `MockAgent`, and `scorer_exact_match` were just names you imported directly. That's fine in one notebook — but it means anyone extending the harness has to edit *your* file.

A **registry** flips this: each plugin type (environment / agent / scorer) gets a dict, and a decorator that populates it *as a side effect of importing the plugin module*. The core harness only ever asks the registry "give me the thing named X" — it never hardcodes what X can be.

This is exactly how `pytest` discovers plugins, how Flask discovers blueprints, and how `sklearn` lets you write a custom estimator that plugs into `GridSearchCV` without touching sklearn's source.

In [ ]:
# registry.py

ENVIRONMENT_REGISTRY: dict[str, type] = {}
AGENT_REGISTRY: dict[str, type] = {}
SCORER_REGISTRY: dict[str, Callable] = {}

def register_environment(name: str):
    def deco(cls):
        if name in ENVIRONMENT_REGISTRY:
            raise ValueError(f"Environment '{name}' already registered by {ENVIRONMENT_REGISTRY[name]!r} — "
                              f"pick a unique name to avoid silently shadowing a built-in")
        ENVIRONMENT_REGISTRY[name] = cls
        return cls
    return deco

def register_agent(name: str):
    def deco(cls):
        if name in AGENT_REGISTRY:
            raise ValueError(f"Agent '{name}' already registered by {AGENT_REGISTRY[name]!r}")
        AGENT_REGISTRY[name] = cls
        return cls
    return deco

def register_scorer(name: str):
    def deco(fn):
        if name in SCORER_REGISTRY:
            raise ValueError(f"Scorer '{name}' already registered by {SCORER_REGISTRY[name]!r}")
        SCORER_REGISTRY[name] = fn
        return fn
    return deco

def get_environment(name: str):
    if name not in ENVIRONMENT_REGISTRY:
        raise KeyError(f"Unknown environment '{name}'. Registered: {sorted(ENVIRONMENT_REGISTRY)}")
    return ENVIRONMENT_REGISTRY[name]

def get_agent(name: str):
    if name not in AGENT_REGISTRY:
        raise KeyError(f"Unknown agent '{name}'. Registered: {sorted(AGENT_REGISTRY)}")
    return AGENT_REGISTRY[name]

def get_scorer(name: str):
    if name not in SCORER_REGISTRY:
        raise KeyError(f"Unknown scorer '{name}'. Registered: {sorted(SCORER_REGISTRY)}")
    return SCORER_REGISTRY[name]

print("registry.py: 3 registries + register_*/get_* pairs. Duplicate-name registration raises loudly (pitfall #1 below).")


In [ ]:
# environments/*.py — built-ins, each self-registers via the decorator on import (recreated from L61 shapes)

@register_environment("calc")
class CalcEnv:
    """A safe-eval calculator tool environment."""
    def __init__(self, task: Task):
        self.task = task

    def call_tool(self, expression: str) -> str:
        allowed = set("0123456789+-*/(). ")
        if not set(expression) <= allowed:
            return "ERROR: disallowed characters"
        try:
            return str(eval(expression, {"__builtins__": {}}, {}))
        except Exception as e:
            return f"ERROR: {e}"

@register_environment("files")
class FileEnv:
    """An in-memory virtual filesystem for edit-then-check tasks (SWE-bench-shaped)."""
    def __init__(self, task: Task):
        self.task = task
        self.fs = dict(task.metadata.get("initial_files", {}))

    def read_file(self, path: str) -> str:
        return self.fs.get(path, f"ERROR: {path} not found")

    def write_file(self, path: str, content: str) -> str:
        self.fs[path] = content
        return "OK"

@register_environment("knowledge")
class KnowledgeEnv:
    """A fixed small corpus + lookup tool (mini-GAIA shape)."""
    CORPUS = {
        "transformer_paper": "Attention Is All You Need (Vaswani et al. 2017) introduced the Transformer architecture.",
        "bert_paper": "BERT (Devlin et al. 2018) introduced masked language modeling pretraining.",
        "lora_paper": "LoRA (Hu et al. 2021) introduced low-rank adaptation for efficient fine-tuning.",
    }
    def __init__(self, task: Task):
        self.task = task

    def lookup(self, key: str) -> str:
        return self.CORPUS.get(key, "ERROR: not found")

print(f"Registered environments: {sorted(ENVIRONMENT_REGISTRY)}")


In [ ]:
# agents/*.py — built-ins

@register_agent("mock")
class MockAgent:
    """Deterministic scripted agent. flake_rate injects controlled randomness for pass@k demos."""
    def __init__(self, script: Optional[dict] = None, flake_rate: float = 0.0):
        self.script = script or {}
        self.flake_rate = flake_rate

    def run(self, task: Task, env) -> Trajectory:
        traj = Trajectory(task_id=task.id)
        if random.random() < self.flake_rate:
            traj.final_state = None
            traj.steps.append(TrajectoryStep(role="agent", content="(flaked)"))
            return traj
        answer = self.script.get(task.id, "no scripted answer")
        traj.steps.append(TrajectoryStep(role="agent", content=answer))
        traj.final_state = answer
        return traj

@register_agent("claude_tool")
class ClaudeToolAgent:
    """Real Sonnet/Haiku tool-use loop. Falls back to a deterministic stub with no API key
    so the harness itself stays free to run and test."""
    def __init__(self, model: str = "claude-sonnet-4-5", max_turns: int = 5):
        self.model = model
        self.max_turns = max_turns

    def run(self, task: Task, env) -> Trajectory:
        traj = Trajectory(task_id=task.id)
        if not HAVE_API_KEY:
            traj.steps.append(TrajectoryStep(role="agent", content="[offline stub — no ANTHROPIC_API_KEY]"))
            traj.final_state = None
            return traj
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model=self.model, max_tokens=256,
            messages=[{"role": "user", "content": task.prompt}],
        )
        text = resp.content[0].text if resp.content else ""
        traj.steps.append(TrajectoryStep(role="agent", content=text))
        traj.final_state = text
        traj.cost_usd = (resp.usage.input_tokens * 3 + resp.usage.output_tokens * 15) / 1_000_000
        return traj

print(f"Registered agents: {sorted(AGENT_REGISTRY)}")


In [ ]:
# scorers/builtin.py

@register_scorer("exact_match")
def scorer_exact_match(task: Task, traj: Trajectory) -> TaskResult:
    expected = task.metadata.get("expected")
    passed = (traj.final_state == expected)
    return TaskResult(task_id=task.id, passed=passed, score=1.0 if passed else 0.0, trajectory=traj)

@register_scorer("unit_test")
def scorer_unit_test(task: Task, traj: Trajectory) -> TaskResult:
    check = task.metadata.get("check")  # callable(final_state) -> bool
    try:
        passed = bool(check(traj.final_state)) if check else False
    except Exception as e:
        return TaskResult(task_id=task.id, passed=False, score=0.0, trajectory=traj, error=str(e))
    return TaskResult(task_id=task.id, passed=passed, score=1.0 if passed else 0.0, trajectory=traj)

@register_scorer("llm_judge")
def scorer_llm_judge(task: Task, traj: Trajectory) -> TaskResult:
    """Deterministic offline fallback: score by keyword overlap with rubric.
    (L61's real version tool-forces Haiku; omitted here to keep this package's core test suite API-free.)"""
    rubric_terms = task.metadata.get("rubric_terms", [])
    text = (traj.final_state or "").lower()
    hits = sum(1 for t in rubric_terms if t.lower() in text)
    score = hits / max(1, len(rubric_terms))
    return TaskResult(task_id=task.id, passed=score >= 0.5, score=score, trajectory=traj)

print(f"Registered scorers: {sorted(SCORER_REGISTRY)}")


In [ ]:
# core.py (cont'd) — BenchmarkRunner, now resolving everything through the registry instead of direct references

class BenchmarkRunner:
    def __init__(self, tasks: list[Task]):
        self.tasks = tasks

    def run(self, agent, k: int = 1) -> list[TaskResult]:
        results = []
        for task in self.tasks:
            env_cls = get_environment(task.env_name)
            scorer_fn = get_scorer(task.scorer_name)
            env = env_cls(task)
            best = None
            for _ in range(k):
                start = time.time()
                traj = agent.run(task, env)
                traj.elapsed_s = time.time() - start
                result = scorer_fn(task, traj)
                if best is None or result.score > best.score:
                    best = result
            results.append(best)
        return results

def aggregate_pass_rate(results: list[TaskResult]) -> float:
    if not results:
        return 0.0
    return sum(1 for r in results if r.passed) / len(results)

print("BenchmarkRunner now resolves environments/scorers via get_environment()/get_scorer() — zero hardcoded imports.")


In [ ]:
# suites/loader.py — a task suite is now data (JSON/YAML), not Python literals baked into a notebook cell.
# This is the concrete unlock from registries: anyone can ship a suite file without touching this package's code.

import io

DEMO_SUITE_JSON = """
[
  {"id": "calc_1", "category": "tool_use", "difficulty": "easy",
   "prompt": "What is 12 * 7?", "env_name": "calc", "scorer_name": "exact_match",
   "metadata": {"expected": "84"}},
  {"id": "calc_2", "category": "tool_use", "difficulty": "easy",
   "prompt": "What is 100 / 4?", "env_name": "calc", "scorer_name": "exact_match",
   "metadata": {"expected": "25.0"}},
  {"id": "know_1", "category": "knowledge_qa", "difficulty": "medium",
   "prompt": "What did the Transformer paper introduce?", "env_name": "knowledge", "scorer_name": "llm_judge",
   "metadata": {"rubric_terms": ["attention", "transformer"]}}
]
"""

def load_suite(source: str) -> list[Task]:
    """Load a task suite from a JSON string, JSON file path, or (if PyYAML is installed) a YAML file path."""
    if source.strip().startswith("["):
        raw = json.loads(source)
    elif source.endswith((".yaml", ".yml")):
        import yaml
        with open(source) as f:
            raw = yaml.safe_load(f)
    else:
        with open(source) as f:
            raw = json.load(f)
    return [Task(**item) for item in raw]

demo_tasks = load_suite(DEMO_SUITE_JSON)
print(f"Loaded {len(demo_tasks)} tasks from an inline JSON suite (same loader path as a suite.yaml file on disk):")
for t in demo_tasks:
    print(f"  - {t.id:10s} env={t.env_name:10s} scorer={t.scorer_name}")


## CLI design

The CLI's job is thin: parse args, load a suite, resolve an agent by name from the registry, run, print a report. All the logic already lives in `core.py` — the CLI is a wrapper, same principle as L58's `paper-distiller` CLI wrapping `distill()`.

In [ ]:
import typer
from typing_extensions import Annotated

cli_app = typer.Typer(rich_markup_mode="rich", help="agent-bench: a pluggable benchmark harness for AI agents.")

@cli_app.command("list-environments")
def cmd_list_environments():
    """List all registered environment plugins."""
    for name in sorted(ENVIRONMENT_REGISTRY):
        typer.echo(f"{name}\t{ENVIRONMENT_REGISTRY[name].__doc__ or ''}".strip())

@cli_app.command("list-agents")
def cmd_list_agents():
    """List all registered agent plugins."""
    for name in sorted(AGENT_REGISTRY):
        typer.echo(f"{name}\t{AGENT_REGISTRY[name].__doc__ or ''}".strip())

@cli_app.command("list-scorers")
def cmd_list_scorers():
    """List all registered scorer plugins."""
    for name in sorted(SCORER_REGISTRY):
        typer.echo(name)

@cli_app.command("run")
def cmd_run(
    suite: Annotated[str, typer.Argument(help="Path to a suite JSON/YAML file, or '-' for the built-in demo suite")],
    agent: Annotated[str, typer.Option(help="Registered agent name")] = "mock",
    k: Annotated[int, typer.Option(help="Attempts per task (for pass@k)")] = 1,
):
    """Run a task suite against an agent and print a pass-rate report."""
    tasks = demo_tasks if suite == "-" else load_suite(suite)
    agent_obj = get_agent(agent)()
    runner = BenchmarkRunner(tasks)
    results = runner.run(agent_obj, k=k)
    rate = aggregate_pass_rate(results)
    typer.echo(f"Ran {len(results)} tasks with agent='{agent}' (k={k}): pass rate = {rate:.0%}")
    for r in results:
        typer.echo(f"  [{'PASS' if r.passed else 'FAIL'}] {r.task_id}  score={r.score:.2f}")
    if rate < 1.0:
        raise typer.Exit(code=1)

print("cli.py defined: run / list-environments / list-agents / list-scorers")


In [ ]:
# CLI smoke test via Typer's CliRunner — same pattern as L58, catches wiring bugs before they hit PyPI.
from typer.testing import CliRunner

runner_cli = CliRunner()

res = runner_cli.invoke(cli_app, ["list-environments"])
assert res.exit_code == 0, res.output
print("$ agent-bench list-environments")
print(res.output)

res = runner_cli.invoke(cli_app, ["run", "-", "--agent", "mock"])
print("$ agent-bench run - --agent mock")
print(res.output)
# exit_code 1 is EXPECTED here: MockAgent has no script wired for calc_1/calc_2/know_1, so it fails all 3 —
# proving the CLI's threshold-based exit code actually works, not that something is broken.
assert res.exit_code == 1
print(f"(exit_code={res.exit_code} as expected — unscripted MockAgent should fail every task)")


In [ ]:
# pyproject.toml — the actual packaging manifest, written to disk exactly as it would ship

pyproject_toml = """
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "agent-bench"
version = "0.1.0"
description = "A pluggable benchmark harness for AI agents: Task, Environment, Agent, Scorer, all extensible via plugins."
readme = "README.md"
requires-python = ">=3.10"
license = {text = "MIT"}
dependencies = [
    "pydantic>=2.0",
    "typer[all]>=0.9",
    "rich>=13.0",
]

[project.optional-dependencies]
claude = ["anthropic>=0.40"]
yaml = ["pyyaml>=6.0"]
dev = ["pytest>=7.0", "pytest-cov", "ruff", "mypy", "build", "twine"]
all = ["agent-bench[claude,yaml,dev]"]

[project.scripts]
agent-bench = "agent_bench.cli:cli_app"

[project.entry-points."agent_bench.environments"]
# Third-party packages can add entries here to register plugins WITHOUT editing agent-bench's source —
# the real payoff of the registry pattern, wired via importlib.metadata.entry_points() in L66.

[project.urls]
Homepage = "https://github.com/gouravkhanijoe/agent-bench"
Issues = "https://github.com/gouravkhanijoe/agent-bench/issues"
"""

import os
os.makedirs("/content/agent_bench_pkg", exist_ok=True)
with open("/content/agent_bench_pkg/pyproject.toml", "w") as f:
    f.write(pyproject_toml)
print("Wrote pyproject.toml — note [project.entry-points.\"agent_bench.environments\"]: this is the real")
print("distribution mechanism for third-party plugins (pip-installed packages, not just files in one repo).")
print("L66 wires this up for real with importlib.metadata.entry_points().")


In [ ]:
# Harness self-test — the load-bearing check from L61, re-run here against the REGISTRY-based runner.
# A zero-flake MockAgent scripted with correct answers must 100%-pass every task before we trust any real
# agent's score. If this fails, the bug is in the harness, not the agent under test.

scripted_answers = {"calc_1": "84", "calc_2": "25.0"}
# know_1 uses llm_judge (keyword rubric) rather than exact_match, so we script text containing the rubric terms:
scripted_answers_full = {**scripted_answers, "know_1": "The Transformer paper introduced the attention mechanism."}

perfect_agent = MockAgent(script=scripted_answers_full, flake_rate=0.0)
runner = BenchmarkRunner(demo_tasks)
results = runner.run(perfect_agent, k=1)
rate = aggregate_pass_rate(results)

table = Table(title="Harness self-test (zero-flake MockAgent)")
table.add_column("task"); table.add_column("passed"); table.add_column("score")
for r in results:
    table.add_row(r.task_id, str(r.passed), f"{r.score:.2f}")
console.print(table)

assert rate == 1.0, f"Harness self-test FAILED: pass rate {rate:.0%}, expected 100%. Bug is in the harness."
print(f"\nSelf-test passed: {rate:.0%} — the registry-based runner is trustworthy before we score anything real.")


In [ ]:
# Extensibility demo — benchmark a THIRD-PARTY-STYLE agent registered from outside this cell's original
# definitions, proving new agents plug in without touching core.py, registry.py, or the CLI.

@register_agent("paper_distiller_stub")
class PaperDistillerAgentStub:
    """Stands in for L61's PaperDistillerAgent — wraps a fixed digest-shaped response, registered as a plugin."""
    def run(self, task: Task, env) -> Trajectory:
        traj = Trajectory(task_id=task.id)
        # Pretend this calls paper_distiller.distill() under the hood (L56-L64's real pipeline).
        canned = "The Transformer paper introduced the attention mechanism and transformer architecture."
        traj.steps.append(TrajectoryStep(role="agent", content=canned))
        traj.final_state = canned
        return traj

assert "paper_distiller_stub" in AGENT_REGISTRY
results2 = BenchmarkRunner([t for t in demo_tasks if t.id == "know_1"]).run(PaperDistillerAgentStub(), k=1)
print(f"paper_distiller_stub on know_1: passed={results2[0].passed}, score={results2[0].score:.2f}")
print("\nNothing in core.py, registry.py, or cli.py changed to make this work — that's the whole point of today's lesson.")


## Pitfalls

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | Silent plugin name collisions | Two packages both register `"calc"` — without the loud `ValueError` in §8, the second import silently shadows the first and scores go wrong with no error |
| 2 | Registry populated only via side-effect imports | If a plugin module is never `import`-ed, it's never registered — `agent-bench run` needs to import all built-ins eagerly in `__init__.py`, not lazily |
| 3 | No isolation for third-party plugins | A malicious or buggy `Environment.call_tool()` runs with the harness's full permissions — same lesson as L63's tool-permission gate, now applied to plugins instead of agents |
| 4 | Suite files trusted blindly | `load_suite()` does no schema validation beyond Pydantic field types — a suite with a nonexistent `env_name` fails at *runtime* mid-benchmark, not at load time; validate `env_name`/`scorer_name` exist in the registry before running any task |
| 5 | `k>1` without `return_exceptions`-style isolation | One flaky/crashing attempt inside the `for _ in range(k)` loop in §12 currently has no try/except — a single exception kills the whole task's scoring instead of just that attempt |
| 6 | Version-pinning plugins to a specific core version | Third-party environment packages can silently break when core's `Task` schema changes — needs a `agent-bench>=0.1,<0.2` style constraint in each plugin's own `pyproject.toml` |
| 7 | CLI exit code semantics undocumented | `cmd_run` exits 1 on `rate < 1.0` — fine for a strict CI gate, wrong for exploratory local runs; needs a `--threshold` option instead of a hardcoded 100% bar |
| 8 | Registries are module-level global state | Two test files that both register a plugin named `"test_env"` will collide across a pytest session unless registries are cleared/scoped per test (`conftest.py` fixture with registry snapshot/restore) |
| 9 | No `list-*` output stability guarantee | `cmd_list_environments` prints `__doc__` directly — a plugin author changing a docstring silently changes CLI output that some downstream script might be parsing; document that CLI text output is not a stable API, only exit codes and (future) `--json` are |
| 10 | Confusing "agent-bench the tool" with "agent-bench the suite" | Today's package is the *harness*; the actual benchmark *content* (task suites) is a separate, evolving artifact — don't bake one org's private eval tasks into the OSS package's built-in suite |
| 11 | `dir()` inside a comprehension only sees the comprehension's own scope | `all(name in dir() for name in [...])` — a genexpr/listcomp creates its own local scope in Python 3, so the bare `dir()` call inside it does **not** see the enclosing cell's globals; caught live while validating this lesson's own verification cell (§21) — fix by snapshotting `dir()` into a variable *before* the comprehension |

Pitfall #3 is the one to sit with: registries make extension easy, and "easy to extend" and "easy to exploit" are the same door. Pitfall #11 is a reminder that even the verification code that's supposed to catch bugs can have its own — that's exactly why every lesson in this curriculum is validated by actually running it, not just read for plausibility.

In [ ]:
# Verification checklist

_current_names = dir()  # snapshot outside the genexpr — dir() inside a comprehension sees only its own local scope
checks = {
    "Task/TrajectoryStep/Trajectory/TaskResult defined": all(name in _current_names for name in
        ["Task", "TrajectoryStep", "Trajectory", "TaskResult"]),
    "pass_at_k importable and correct at boundary (n=c=k=1)": pass_at_k(1, 1, 1) == 1.0,
    "3 registries populated": len(ENVIRONMENT_REGISTRY) >= 3 and len(AGENT_REGISTRY) >= 3 and len(SCORER_REGISTRY) >= 3,
    "duplicate registration raises ValueError": False,
    "load_suite() parses inline JSON": len(demo_tasks) == 3,
    "CLI list-environments exits 0": True,
    "CLI run exits 1 on imperfect unscripted agent (threshold works)": True,
    "harness self-test hit 100% with a correctly-scripted MockAgent": rate == 1.0,
    "third-party-style agent plugs in with zero core edits": results2[0].passed == True,
    "pyproject.toml written to disk": os.path.exists("/content/agent_bench_pkg/pyproject.toml"),
}

# Actually exercise the duplicate-registration check rather than assuming it:
try:
    @register_environment("calc")
    class _Dup:
        pass
    checks["duplicate registration raises ValueError"] = False
except ValueError:
    checks["duplicate registration raises ValueError"] = True

table = Table(title="Lesson 65 verification")
table.add_column("check"); table.add_column("passed")
all_ok = True
for name, ok in checks.items():
    table.add_row(name, "PASS" if ok else "FAIL")
    all_ok = all_ok and ok
console.print(table)
assert all_ok, "One or more verification checks failed — see table above."
print("\nAll checks passed.")


## Summary

| Concept | One-line takeaway |
|---|---|
| Repackaging a proven component | `agent-bench`'s Task/Environment/Agent/Scorer contract was already validated across 3 agent types in L61 — today's work was packaging, not re-proving the idea |
| Plugin registry pattern | A small core + `register_*`/`get_*` dict pairs decouples "the harness" from "the things it benchmarks" — same idea as pytest plugins, Flask blueprints, sklearn estimators |
| Loud collision detection | Duplicate plugin names raise `ValueError` immediately rather than silently shadowing — cheap insurance against a nasty class of bug |
| Suites as data, not code | `load_suite()` turns a JSON/YAML file into `list[Task]` — anyone can ship a new benchmark suite without touching this package's source |
| CLI as a thin wrapper | `cli.py` has zero business logic — it resolves names via the registry and calls `core.py`, mirroring L58's paper-distiller CLI |
| `entry_points` as the real distribution story | `[project.entry-points."agent_bench.environments"]` in `pyproject.toml` is how *pip-installed* third-party plugins register themselves — stubbed today, wired for real in L66 |
| Harness self-test before trusting scores | A zero-flake, correctly-scripted `MockAgent` must 100%-pass every task — re-run after ANY change to core, exactly as in L61 |
| Extension without core edits, proven not assumed | `PaperDistillerAgentStub` was registered and benchmarked without touching `core.py`/`registry.py`/`cli.py` — the verification checklist actually asserts this rather than taking it on faith |

### Homework
1. Add a 4th built-in environment: `ShellEnv`, a sandboxed subprocess runner (Terminal-Bench-shaped) — this is L66's topic, but sketch your own version first.
2. Write a `--json` output mode for `cmd_run` so downstream tooling has something more stable to parse than the pitfall-#9 doc-string text.
3. Add per-attempt exception isolation inside `BenchmarkRunner.run()`'s `for _ in range(k)` loop (pitfall #5) and write a test that a crashing attempt doesn't kill the whole task.
4. Move the built-in environments/agents/scorers into real separate files (`environments/calc.py` etc.) matching the tree in §5, and confirm `import agent_bench` still populates all 3 registries via `__init__.py`'s eager imports.
5. Write a `conftest.py` fixture that snapshots and restores all 3 registries around each test (pitfall #8), then write two tests that each register a plugin named `"test_env"` and confirm they don't collide across the test session.

### Next lesson (66): Custom Environments & Third-Party Plugin Discovery
We'll build `ShellEnv` for real (sandboxed subprocess execution — the actual Terminal-Bench shape), then wire up `importlib.metadata.entry_points()` so a *separately pip-installed* package can register a plugin into `agent-bench` with zero shared source, proving the `[project.entry-points]` stub from §17 for real.

**Phase 7, Lesson 1 of ~6, complete.**